# Verification of Brazilian Municipality Centroids — GEE x IBGE

**Goal:** load the table of Brazilian municipality centroids provided as an asset on Google Earth Engine (`projects/fcoliveira/assets/centroide_br`), check its coverage against the official municipality list from the IBGE API (5,570 municipalities), and plot the points on a map.

**Notebook structure:**
1. Installation and authentication (GEE)
2. Loading the asset and inspecting the available fields
3. Extracting the municipality codes from the asset
4. Querying the IBGE API (official municipality list)
5. Comparison: missing municipalities / unmatched codes
6. Reports (CSV) and map visualization (geemap)

> ⚠️ **Note:** the column structure of the asset (`centroide_br`) is not known ahead of time. Cell 3 inspects the properties of the first feature — adjust the `COL_CODIGO` variable in the next cell with the correct name of the field that holds the municipality's 7-digit IBGE code.

In [26]:
# 1. Install the required libraries (run once; in Colab it may require restarting the runtime the first time)
!pip install earthengine-api geemap pandas requests geopandas -q

In [27]:
import ee
# The Folium/Leaflet backend does not look up GOOGLE_MAPS_API_KEY on Colab.
import geemap.foliumap as geemap
import pandas as pd
import requests

pd.set_option('display.max_rows', 100)

## 1. Earth Engine authentication and initialization

Replace `SEU_PROJETO_GEE` with the ID of your Google Cloud project linked to Earth Engine (required since the migration to project-based authentication).

In [28]:
GEE_PROJECT = 'SEU_PROJETO_GEE'  # <-- SET THIS: your GEE/Cloud project ID

try:
    ee.Initialize(project='projeto_no_GEE')
except Exception:
    ee.Authenticate()
    ee.Initialize(project='projeto_no_GEE')

print('Earth Engine inicializado com sucesso.')

Earth Engine inicializado com sucesso.


## 2. Loading the asset and inspecting its structure

In [29]:
ASSET_ID = 'projects/fcoliveira/assets/centroide_br'
centroides = ee.FeatureCollection(ASSET_ID)

n_features = centroides.size().getInfo()
print(f'Total de feições (pontos) no asset GEE: {n_features}')
print('Total oficial de municípios: será obtido diretamente da API do IBGE na etapa 4.')

Total de feições (pontos) no asset GEE: 5571
Total oficial de municípios: será obtido diretamente da API do IBGE na etapa 4.


In [30]:
# Inspect the properties of the first feature to identify the field with the IBGE code
primeira = centroides.first()
props = primeira.getInfo()['properties']

print('Propriedades disponíveis no asset:')
for k, v in props.items():
    print(f'  {k!r}: {v!r}')

Propriedades disponíveis no asset:
  'mesorregiao': ''
  'microrregiao': ''
  'nome_municipio': 'Boa Esperança do Norte'
  'regiao_imediata': 'Sorriso'
  'regiao_intermediaria': 'Sinop'
  'regiao_nome': ''
  'regiao_sigla': ''
  'uf_nome': ''
  'uf_sigla': ''
  '\ufeffcodigo_ibge': 5101837


## 3. Extracting the municipality codes from the asset

Adjust `COL_CODIGO` below according to the field name identified in the previous cell (e.g. `CD_MUN`, `CD_GEOCMU`, `GEOCODIGO`, `codigo_ibge`, `id`, etc.). The IBGE municipality code has 7 digits.

In [31]:
# The current asset has a BOM (\ufeff) before codigo_ibge.
COL_CODIGO = '\ufeffcodigo_ibge'

codigos_gee_raw = centroides.aggregate_array(COL_CODIGO).getInfo()
codigos_gee = [str(c).strip() for c in codigos_gee_raw]

print(f'Total de códigos extraídos do GEE: {len(codigos_gee)}')
print(f'Códigos duplicados no asset GEE: {len(codigos_gee) - len(set(codigos_gee))}')
print('Exemplos:', codigos_gee[:5])

Total de códigos extraídos do GEE: 5571
Códigos duplicados no asset GEE: 0
Exemplos: ['5101837', '1200054', '1200104', '1200252', '1200708']


## 4. Official municipality list — IBGE API

Endpoint: `https://servicodados.ibge.gov.br/api/v1/localidades/municipios`

In [32]:
url_ibge = 'https://servicodados.ibge.gov.br/api/v1/localidades/municipios'
resp = requests.get(url_ibge, timeout=60)
resp.raise_for_status()
municipios_ibge = resp.json()

df_ibge = pd.json_normalize(municipios_ibge)
df_ibge['codigo_ibge'] = df_ibge['id'].astype(str)
df_ibge['nome_municipio'] = df_ibge['nome']
df_ibge['uf'] = df_ibge['microrregiao.mesorregiao.UF.sigla']

df_ibge = df_ibge[['codigo_ibge', 'nome_municipio', 'uf']].sort_values(['uf', 'nome_municipio']).reset_index(drop=True)

print(f'Total de municípios segundo a API do IBGE: {len(df_ibge)}')
df_ibge.head()

Total de municípios segundo a API do IBGE: 5571


,codigo_ibge,nome_municipio,uf
0,1200013,Acrelândia,AC
1,1200054,Assis Brasil,AC
2,1200104,Brasiléia,AC
3,1200138,Bujari,AC
4,1200179,Capixaba,AC


## 5. GEE x IBGE comparison

In [33]:
set_gee = set(codigos_gee)
set_ibge = set(df_ibge['codigo_ibge'])

faltantes_no_gee = set_ibge - set_gee     # IBGE municipalities that are NOT in the GEE asset
extras_no_gee = set_gee - set_ibge        # codes in the GEE asset with no match in IBGE (possible error/duplicate)

print(f'Municípios do IBGE ausentes no asset GEE: {len(faltantes_no_gee)}')
print(f'Códigos no GEE sem correspondência na lista do IBGE: {len(extras_no_gee)}')

if len(set_gee) == len(codigos_gee) and len(faltantes_no_gee) == 0 and len(extras_no_gee) == 0:
    print('\n✅ Cobertura completa: todos os 5.570 municípios estão presentes e sem duplicidade.')

Municípios do IBGE ausentes no asset GEE: 0
Códigos no GEE sem correspondência na lista do IBGE: 0

✅ Cobertura completa: todos os 5.570 municípios estão presentes e sem duplicidade.


In [34]:
df_faltantes = df_ibge[df_ibge['codigo_ibge'].isin(faltantes_no_gee)].reset_index(drop=True)
print(f'{len(df_faltantes)} município(s) faltando no asset GEE:')
df_faltantes

0 município(s) faltando no asset GEE:


,codigo_ibge,nome_municipio,uf


In [35]:
df_extras = pd.DataFrame(sorted(extras_no_gee), columns=['codigo_gee_sem_correspondencia'])
print(f'{len(df_extras)} código(s) no asset GEE sem correspondência na base do IBGE:')
df_extras

0 código(s) no asset GEE sem correspondência na base do IBGE:


,codigo_gee_sem_correspondencia


## 6. CSV reports

In [36]:
if len(df_faltantes) > 0:
    df_faltantes.to_csv('municipios_faltantes.csv', index=False, encoding='utf-8-sig')
    print('Relatório de municípios faltantes salvo em: municipios_faltantes.csv')
else:
    print('Nenhum município faltando — relatório não gerado.')

if len(df_extras) > 0:
    df_extras.to_csv('codigos_extras_gee.csv', index=False, encoding='utf-8-sig')
    print('Relatório de códigos extras salvo em: codigos_extras_gee.csv')

Nenhum município faltando — relatório não gerado.


## 7. Map visualization (geemap)

Displays all the points from the GEE asset. Hover / click on the points to see the original attributes.

In [37]:
# Folium/Leaflet avoids the GOOGLE_MAPS_API_KEY dependency on Colab.
Map = geemap.Map(location=[-15, -52], zoom_start=4)

vis_params = {'color': '1f77b4', 'pointSize': 3}
Map.add_layer(centroides.style(**{'color': '1f77b4', 'pointSize': 3}), {}, 'Centroides de Municípios (GEE)')

Map.add_layer_control()
Map

## 8. (Optional/advanced) Locate the geometry of missing municipalities via IBGE

The basic IBGE API list does not return coordinates. To plot exactly where the missing municipalities are, it's possible to fetch the geographic mesh of each one individually (`/api/v3/malhas/municipios/{codigo}`) and compute the centroid locally. This makes one request per missing municipality — only recommended if the missing list is small.

In [38]:
# Only run this if there are missing municipalities and the list is small (e.g. < 50)
RODAR_BUSCA_GEOMETRIA = False  # <-- change to True to run

if RODAR_BUSCA_GEOMETRIA and len(df_faltantes) > 0:
    import geopandas as gpd
    from shapely.geometry import shape

    registros = []
    for codigo in df_faltantes['codigo_ibge']:
        url = f'https://servicodados.ibge.gov.br/api/v3/malhas/municipios/{codigo}?formato=application/vnd.geo+json'
        r = requests.get(url, timeout=60)
        if r.status_code == 200:
            geojson = r.json()
            geom = shape(geojson['features'][0]['geometry'])
            centroide = geom.centroid
            registros.append({'codigo_ibge': codigo, 'lon': centroide.x, 'lat': centroide.y})

    df_faltantes_geo = pd.DataFrame(registros).merge(df_faltantes, on='codigo_ibge')
    print(df_faltantes_geo)

    # Adds the missing points to the map in red, for visual inspection
    faltantes_fc = ee.FeatureCollection([
        ee.Feature(ee.Geometry.Point([row.lon, row.lat]), {'nome_municipio': row.nome_municipio, 'uf': row.uf})
        for row in df_faltantes_geo.itertuples()
    ])
    Map.add_layer(faltantes_fc.style(**{'color': 'd62728', 'pointSize': 6}), {}, 'Municípios FALTANTES (vermelho)')
    Map
else:
    print('Etapa não executada (ajuste RODAR_BUSCA_GEOMETRIA e/ou não há faltantes).')

Etapa não executada (ajuste RODAR_BUSCA_GEOMETRIA e/ou não há faltantes).


## Summary

- Total in the GEE asset: see cell 2
- Official IBGE total: 5,570 municipalities
- Missing: `df_faltantes`
- Duplicates/extra codes: `df_extras`

If `df_faltantes` and `df_extras` are empty and there is no duplication in `codigos_gee`, the asset's coverage is complete and consistent with the official IBGE database.